In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "examples" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())


# K562真实参考输入：免GPU重建Top10



仓库提供已有K562实验运行的紧凑输入：11算法×100条来源记录、对应的11 PWM和已通过阈值/NMS的FIMO位点。已经保存位点，因此可以独立重算整个选择过程：



In [ ]:
from pathlib import Path
import pandas as pd
from sequence_optimization.motifs import read_meme, select_motif_panel

DATA = Path("examples/data")
records = pd.read_csv(DATA / "k562_candidates_1100.csv")
hits = pd.read_csv(DATA / "k562_fimo_sites.csv")
catalog = read_meme(DATA / "k562_catalog.meme")
assert len(records) == 1100
selected, trace = select_motif_panel(
    records, hits, catalog,
    output_dir="outputs/reference_k562_top10_run1",
    top_n=10, min_score=20., min_sites=30,
)
print(selected[["selection_rank", "sequence_id", "score", "present_motifs"]])




这些PWM仅用于复现K562参考流程；不能据此宣称promoter、UTR或小鼠也有相同motif。位点表的q值来自原始完整catalog扫描，未在打包时重算。fixture重建了稳定SHA256序列ID，旧结果的`U_...`ID通过DNA重新映射，DNA和预测数值保持不变。

输出包括：`selected_sequences.fa/csv`、唯一及来源记录表、motif位点与支持数、唯一DNA和全部记录presence matrix、PWM相似矩阵、composition/position/count距离NPZ、逐步选择trace和参数receipt。NPZ始终保存行顺序ID，不依赖CSV默认行号。


比较已保存的参考序列及顺序：



In [ ]:
expected = pd.read_csv(DATA / "k562_expected_top10.csv")
assert selected.sequence.tolist() == expected.sequence.tolist()
assert selected.sequence_id.tolist() == expected.sequence_id.tolist()
print("PASS: exact Top10 DNA and selection order reproduced")
